In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing import text
from tensorflow.keras.utils import pad_sequences, to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, Lambda
import tensorflow.keras.backend as K

In [2]:
data = [
    "Natural Language Processing is a field of Artificial Intelligence.",
    "Word embeddings help computers understand human language.",
    "The CBOW model is a part of Word2Vec technique.",
    "CBOW predicts the target word using surrounding context words.",
    "Skip Gram is another architecture of Word2Vec.",
    "Word2Vec is widely used in NLP applications.",
    "Embedding layers in deep learning are used to represent words.",
    "CBOW is faster and works better with frequent words."
]

tokenizer = text.Tokenizer()
tokenizer.fit_on_texts(data)

word2id = tokenizer.word_index
word2id['PAD'] = 0
id2word = {v:k for k,v in word2id.items()}

wids = [[word2id[w] for w in text.text_to_word_sequence(s)] for s in data]

vocab_size = len(word2id)
window_size = 2
embed_size = 100

In [3]:
def generate_data(corpus):
    for sentence in corpus:
        for i, target in enumerate(sentence):
            context = [sentence[j] for j in range(max(0,i-2), min(len(sentence),i+3)) if j != i]
            x = pad_sequences([context], maxlen=4)
            y = to_categorical([target], num_classes=vocab_size)
            yield x, y

In [4]:
cbow = Sequential([
    Embedding(vocab_size, embed_size, input_length=4),
    Lambda(lambda x: K.mean(x, axis=1)),
    Dense(vocab_size, activation='softmax')
])

cbow.compile(loss='categorical_crossentropy', optimizer='adam')

E:\anacd\envs\tf_env\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [5]:
for epoch in range(10):
    loss = 0
    for x, y in generate_data(wids):
        loss += cbow.train_on_batch(x, y)
    print("Epoch:", epoch+1, "Loss:", loss)


Epoch: 1 Loss: 262.5508568286896
Epoch: 2 Loss: 258.506667137146
Epoch: 3 Loss: 254.6198925971985
Epoch: 4 Loss: 249.74007415771484
Epoch: 5 Loss: 243.5673952102661
Epoch: 6 Loss: 235.9750940799713
Epoch: 7 Loss: 227.05667281150818
Epoch: 8 Loss: 217.09016370773315
Epoch: 9 Loss: 206.4199562072754
Epoch: 10 Loss: 195.33682346343994


In [8]:
sentence = input("Enter sentence: ").lower()
words = text.text_to_word_sequence(sentence)

pos = int(input("Enter target word position: ")) - 1

context = [word2id[words[i]] for i in range(max(0,pos-2), min(len(words),pos+3))
           if i != pos and words[i] in word2id]

x = pad_sequences([context], maxlen=4)
pred = cbow.predict(x, verbose=0)
pred[0][0] = 0

word = np.argmax(pred[0])
print("Predicted word:", id2word[word])

Enter sentence:  CBOW predicts the target
Enter target word position:  3


Predicted word: the
